# Enhanced Allergen Labeling & Annotation

**Purpose:** This notebook processes and analyzes food label data for allergen detection. Built on Refactored utility modules for improved maintainability and reduced code duplication.

In [1]:
import os
import pandas as pd
import re

# Import standardized utility modules
from utils.text_processing import (
    clean_html,
    clean_ingredient_text,
    get_allergen_list,
    parse_label_string,
    rule_match,
    detect_allergens_rule_based,
    tokenize_ingredients,
    extract_may_contain,
    has_explicit_allergen_statement,
    parse_traces_tags,
    parse_official_tags,
    apply_exemptions,
    detect_coconut_improved,
    combine_allergen_labels,
    ALLERGEN_RULES,
    OFFICIAL_MAP
)
from utils.data_utils import (
    get_data_directories,
    load_cleaned_data,
    load_labeled_data,
    parse_label_column
)
from utils.model_utils import (
    load_model_and_tokenizer,
    load_hybrid_config,
    predict_ml,
    hybrid_predict,
    find_best_thresholds
)
from utils.evaluation import (
    print_classification_report,
    compute_per_class_metrics,
    jaccard
)

In [2]:
import ast

# Get project directories for consistent path handling
dirs = get_data_directories()
print(f"Project base directory: {dirs['base']}")

# Load the cleaned dataset from the previous step
df = load_cleaned_data(filepath=os.path.join(dirs['processed'], 'cleaned_dataset.csv'))
print(f"Cleaned dataset shape: {df.shape}")

# ── Enrich with rule-based allergen detection ──

# Tokenize ingredient text (full text, not just the cleaned column)
df["ingredient_tokens"] = df["ingredients_text_en"].apply(
    lambda x: tokenize_ingredients(clean_html(x))
)

# Rule-based detection on full ingredient text
df["detected_allergens"] = df["ingredients_text_en"].apply(
    lambda x: detect_allergens_rule_based(str(x).lower())
)

# Parse official allergen tags (from OpenFoodFacts)
df["official_allergens_mapped"] = df["allergens_tags"].apply(
    lambda x: parse_official_tags(x) if pd.notna(x) else []
)

# Parse traces tags
df["traces_allergens"] = df["traces_tags"].apply(
    lambda x: parse_traces_tags(x) if pd.notna(x) else []
)

# Extract "may contain" statements
df["may_contain"] = df["ingredients_text_en"].apply(
    lambda x: extract_may_contain(str(x))
)

# Detect coconut from ingredient tokens
df["coconut"] = df["ingredient_tokens"].apply(
    lambda x: detect_coconut_improved(x) if isinstance(x, list) else []
)

# Build consensus (intersection of detected and official)
def get_consensus(row):
    detected = set(row["detected_allergens"]) if isinstance(row["detected_allergens"], list) else set()
    official = set(row["official_allergens_mapped"]) if isinstance(row["official_allergens_mapped"], list) else set()
    return sorted(detected & official)

df["consensus_allergens"] = df.apply(get_consensus, axis=1)

print(f"\nEnriched dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nSample detected allergens:\n{df[['code', 'detected_allergens', 'official_allergens_mapped']].head(3).to_string()}")

Project base directory: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION
Cleaned dataset shape: (1057, 13)

Enriched dataset shape: (1057, 19)
Columns: ['code', 'brands', 'product_name_en', 'ingredients_text_en', 'ingredients', 'ingredients_tags', 'allergens_tags', 'traces_tags', 'categories', 'countries_tags', 'ingredients_cleaned', 'ingredient_tokens_raw', 'ingredient_tokens', 'detected_allergens', 'official_allergens_mapped', 'traces_allergens', 'may_contain', 'coconut', 'consensus_allergens']

Sample detected allergens:
            code detected_allergens official_allergens_mapped
0  8850389100684                 []                        []
1  4800194105958            [wheat]                        []
2  4806533726822      [milk, wheat]                        []


## 4. Process Official Allergen Tags

Using utility functions for official tag parsing and mapping

In [3]:
# Load hybrid configuration (generated by Notebook 08, may not exist yet)
# The hybrid config is loaded here for completeness but is only consumed
# downstream in Notebook 08 (hybrid_evaluation). Default thresholds are used
# when the file doesn't exist yet.
HYBRID_CONFIG_PATH = "../models/hybrid_config.json"

try:
    hybrid_config = load_hybrid_config(HYBRID_CONFIG_PATH)
    ml_thresholds = hybrid_config["ml_thresholds"]
    rule_conf_threshold = hybrid_config["rule_conf_threshold"]
    mode = hybrid_config["mode"]
    print(f"Loaded ML thresholds: {ml_thresholds}")
    print(f"Rule conf threshold: {rule_conf_threshold}")
    print(f"Mode: {mode}")
except FileNotFoundError:
    print("Hybrid config not found (expected — Notebook 08 generates it later).")
    print("Using default thresholds. These are not used in this notebook.")
    ml_thresholds = [0.5] * 8
    rule_conf_threshold = 0.5
    mode = "rule_priority"

Loaded ML thresholds: [0.05, 0.02, 0.35000000000000003, 0.5, 0.73, 0.13, 0.08, 0.5]
Rule conf threshold: 0.5
Mode: hard_override


## 5. Apply Exemptions

Using utility function for exemption handling

In [4]:
# Create tokens from ingredient text if they don't already exist
# (labeled_dataset_enhanced.csv does not include token columns)
df["ingredient_tokens"] = df["ingredients_text_en"].apply(
    lambda x: tokenize_ingredients(clean_html(x))
)

# Parse string representations of lists to actual Python lists
# (CSV stores list columns as strings like "['wheat', 'milk', 'coconut']")
def parse_detected(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []
df["detected_allergens"] = df["detected_allergens"].apply(parse_detected)
df["official_allergens_mapped"] = df["official_allergens_mapped"].apply(parse_detected)
df["traces_allergens"] = df["traces_allergens"].apply(parse_detected)
df["may_contain"] = df["may_contain"].apply(parse_detected)
df["coconut"] = df["coconut"].apply(parse_detected)
df["consensus_allergens"] = df["consensus_allergens"].apply(parse_detected)

# Apply exemptions using standardized utility function
df["detected_allergens"] = df.apply(
    lambda row: apply_exemptions(row["detected_allergens"], row["ingredient_tokens"]), axis=1
)

print(f"Exemptions applied using standardized utility function.")
print(f"Updated {len(df)} rows with exemption-aware allergen detection.")

Exemptions applied using standardized utility function.
Updated 1057 rows with exemption-aware allergen detection.


## 6. Identify Products with Explicit Allergen Statements

Using utility function for explicit statement detection

In [5]:
# Identify products that explicitly list allergens (more reliable ground truth)
df["explicit_allergen_statement"] = df["ingredients_text_en"].apply(has_explicit_allergen_statement)
eval_df = df[df["explicit_allergen_statement"]]

if len(eval_df) > 0:
    print(f"Evaluating on {len(eval_df)} products with explicit allergen statements (out of {len(df)} total).\n")
    
    strict = (eval_df["detected_allergens"].apply(set) == eval_df["official_allergens_mapped"].apply(set)).mean()
    subset = (eval_df["detected_allergens"].apply(set) <= eval_df["official_allergens_mapped"].apply(set)).mean()
    superset = (eval_df["detected_allergens"].apply(set) >= eval_df["official_allergens_mapped"].apply(set)).mean()
    
    avg_jaccard = eval_df.apply(lambda row: jaccard(set(row["detected_allergens"]), set(row["official_allergens_mapped"])), axis=1).mean()
    
    print(f"Exact match agreement (strict, filtered): {strict:.2%}")
    print(f"Detected ⊆ official (no false positives): {subset:.2%}")
    print(f"Detected ⊇ official (no false negatives): {superset:.2%}")
    print(f"Average Jaccard similarity: {avg_jaccard:.2%}")
else:
    print("No products with explicit allergen statements found.")

Evaluating on 96 products with explicit allergen statements (out of 1057 total).

Exact match agreement (strict, filtered): 21.88%
Detected ⊆ official (no false positives): 21.88%
Detected ⊇ official (no false negatives): 100.00%
Average Jaccard similarity: 22.40%


## Hybrid configuration loaded in Section 4 above

The hybrid config (ML thresholds, rule confidence threshold, and override mode) was loaded earlier in Section 4 and is reused throughout this notebook for evaluation and labeling logic.

In [6]:
# Use combine_allergen_labels from utils.text_processing
# (imported above — uses proper set subtraction for detected_only)

# Apply to dataframe
df["combined_allergens"] = df.apply(
    lambda row: combine_allergen_labels(
        row["detected_allergens"],
        row["official_allergens_mapped"],
        row["traces_allergens"],
        row["may_contain"]
    ), axis=1
)

# Extract consensus (only where both detected and official agree)
df["consensus_allergens"] = df["combined_allergens"].apply(lambda x: x["consensus"])

print("Allergen sources combined using standardized utility function.")

Allergen sources combined using standardized utility function.


## 8. Compute Final Evaluation Metrics

Using standardized evaluation functions

In [7]:
from utils.data_utils import labels_to_binary

# Convert variable-length allergen lists to binary matrix (n_samples x n_classes)
all_allergens = get_allergen_list()
y_true_binary = labels_to_binary(df["official_allergens_mapped"].tolist(), all_allergens)
y_pred_binary = labels_to_binary(df["detected_allergens"].tolist(), all_allergens)

# Evaluate against official tags using standardized functions
strict_agreement = (df["detected_allergens"].apply(set) == df["official_allergens_mapped"].apply(set)).mean()
print(f"Exact match agreement (strict): {strict_agreement:.2%}")

subset_agreement = (df["detected_allergens"].apply(set) <= df["official_allergens_mapped"].apply(set)).mean()
print(f"Detected ⊆ official (no false positives): {subset_agreement:.2%}")

superset_agreement = (df["detected_allergens"].apply(set) >= df["official_allergens_mapped"].apply(set)).mean()
print(f"Detected ⊇ official (no false negatives): {superset_agreement:.2%}")

avg_jaccard = df.apply(lambda row: jaccard(set(row["detected_allergens"]), set(row["official_allergens_mapped"])), axis=1).mean()
print(f"Average Jaccard similarity: {avg_jaccard:.2%}")

# Per-allergen metrics using standardized function
print(f"\nAllergen list from utility module: {all_allergens}")

print("\nPer-allergen metrics:")
metrics = compute_per_class_metrics(y_true_binary, y_pred_binary, all_allergens)

# Print detailed metrics
for allergen in all_allergens:
    m = metrics[allergen]
    print(f"{allergen:12s}  Prec: {m['precision']:.2%}  Rec: {m['recall']:.2%}  F1: {m['f1']:.2%}")

Exact match agreement (strict): 48.44%
Detected ⊆ official (no false positives): 51.09%
Detected ⊇ official (no false negatives): 97.16%
Average Jaccard similarity: 49.62%

Allergen list from utility module: ['milk', 'eggs', 'peanuts', 'tree_nuts', 'soy', 'wheat', 'fish', 'shellfish']

Per-allergen metrics:
milk          Prec: 27.77%  Rec: 96.50%  F1: 43.12%
eggs          Prec: 4.08%  Rec: 100.00%  F1: 7.84%
peanuts       Prec: 14.77%  Rec: 100.00%  F1: 25.74%
tree_nuts     Prec: 31.25%  Rec: 55.56%  F1: 40.00%
soy           Prec: 6.47%  Rec: 85.71%  F1: 12.04%
wheat         Prec: 11.27%  Rec: 83.33%  F1: 19.85%
fish          Prec: 2.60%  Rec: 16.67%  F1: 4.49%
shellfish     Prec: 0.00%  Rec: 0.00%  F1: 0.00%


## 9. Save Final Dataset

Using standardized data utilities

In [8]:
# Create output directory if it doesn't exist
os.makedirs(dirs['final'], exist_ok=True)

# Select columns for final output
final_columns = [
    "code", "brands", "product_name_en", "ingredients_text_en",
    "detected_allergens", "official_allergens_mapped", "traces_allergens",
    "consensus_allergens", "combined_allergens", "may_contain", "coconut"
]
existing_final = [c for c in final_columns if c in df.columns]
final_df = df[existing_final].copy()

# Save using utility data functions
final_path = os.path.join(dirs['final'], 'labeled_dataset_enhanced.csv')
final_df.to_csv(final_path, index=False)
print(f"Saved enhanced dataset with consensus labels to {final_path}")

# Optional: Save metadata using utility function
metadata = {
    "dataset_source": "enhanced_allergen_detection",
    "processed_date": pd.Timestamp.now().isoformat(),
    "num_samples": len(final_df),
    "all_allergens": all_allergens,
    "processing_method": "standardized_utility_modules",
    "notebook_version": "06_allergen_labeling_v2.2.0"
}
metadata_path = os.path.join(dirs['final'], 'processing_metadata.json')
from utils.data_utils import save_metadata
save_metadata(metadata, metadata_path)
print(f"Saved processing metadata to {metadata_path}")

Saved enhanced dataset with consensus labels to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/labeled_dataset_enhanced.csv
Saved processing metadata to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/data/final/processing_metadata.json
